# Get location with RORS

Dans les données stockés dans le fichier `2026-03-11_author_affil_by_publi.csv`, il y a deux cas de figures :

1. des affiliations avec ror
2. des affiliations sans ror


Dans ce notebook, on se concentre sur le premier cas de figure, plus simple à résoudre. En effet, à partir du ror, on peut facilement les informations géographiques en requêtant l'API du Research Organisation Registry (ROR).

Commencçons par charger les librairies dont on a besoin ainsi que les données.

In [1]:
#chargement des modules
import pandas as pd # pour le dataframe

import json # pour lire les données de l'api au format json

import numpy as np #outils mathématiques
import re #regex

In [2]:
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

In [3]:
import requests
from requests.exceptions import HTTPError
import urllib.parse
import tqdm

In [4]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_author_affil_by_publi.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep=",")


In [ ]:
df0

# Transformation du dataframe original

Pour un même auteur, il peut y avoir plus affiliations. Nous allons donc "exploser" le dataframe `df0` pour avoir une ligne par ror

In [ ]:
df0["row_id"] = df0.id +"_"+df0.index.astype(str)
df01 = df0[["id","full_name","row_id","rors"]].copy() # tableau pour prendre uniquement les rors
df01["ror"] = df01.rors.str.split("|")
df_rors = df01.explode("ror").reset_index().drop(columns=["index"])
df_rors0 = df_rors.loc[df_rors.ror =="no ror"]
df_rors1 = df_rors.loc[df_rors.ror !="no ror"]
list_affil_without_ror = [x for x in df_rors0.row_id.unique()]
len(list_affil_without_ror)



# Récupération des informations géographique à partir du ror :

In [ ]:
ROR_API_ENDPOINT = "https://api.dev.ror.org/organizations" # url de l'api ROR


Exemple de requête avec le ror de Rennes 2

In [ ]:
query_response = requests.get(ROR_API_ENDPOINT + '/' + "01m84wm78").json()
#décommenter la ligne ci-dessous pour afficher le résultat
#query_response

Dans le script ci-dessous, on récupère les données géographiques (nom de la ville, du pays, de l'institution, coordonnées GPS)

In [ ]:
dict_ror = {}
list_row = []
for ror in tqdm.tqdm(df_rors1.ror.unique(), total = df_rors1.ror.nunique()):
    dict_row ={}
    rorid = ror.split("/")[-1]
    #print(rorid)
    dict_ror[ror] = rorid
    dict_row["ror"] = ror
    data = requests.get(ROR_API_ENDPOINT + '/' + rorid).json()

    for n, x in enumerate(data["names"]):
        if 'ror_display' in x["types"]:
            dict_row["nom"] = x["value"]
    locations_data = data["locations"][0]["geonames_details"]
    for i in locations_data:
        dict_row[i] = locations_data[i]
        if i == "name":
            dict_row["city"] = locations_data[i]
    dict_row["geonames_id"] = data["locations"][0]["geonames_id"]
    list_row.append(dict_row)


        
    
    
    

In [ ]:
df_inst = pd.DataFrame.from_dict(list_row)
df_inst.columns

In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_ror_with_geographic_info.csv", "w") as file_out:
    df_inst.to_csv(file_out, sep=",", index= False)

In [ ]:
df_rors1 = df_rors1.merge(df_inst[['ror', 'nom', 
       'country_name', 'city', 'geonames_id']], on= ["ror"], how = "left")
df_rors1

In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-20_affiliation_with_info.csv", "w") as file_out:
    df_rors1.to_csv(file_out, sep=",", index= False)